# Import Libraries

In [ ]:
import cv2
import numpy as np
import random
import torch
from scipy.ndimage import gaussian_filter
from skimage.draw import bezier_curve
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Convert Tensor to Numpy and Vice Versa

In [ ]:
def to_numpy(img):
    if isinstance(img, torch.Tensor):
        if img.dim() == 4: img = img.squeeze(0)
        if img.shape[0] == 3: img = img.permute(1, 2, 0)
        return img.cpu().numpy() * 255.0
    return img.astype(np.float32)

def to_tensor(img, device='cpu'):
    # Ensure correct dtype and scaling
    return torch.tensor(img, dtype=torch.float32).permute(2, 0, 1).to(device) / 255.0

# Changes Range of Color Channels

In [ ]:
def change_range_colors(image, min_vals=(0, 0, 0), max_vals=(255, 255, 255)):
    # Split the image into its BGR channels
    b, g, r = cv2.split(image)
    # Clip each channel to its respective range
    b = np.clip(b, min_vals[0], max_vals[0])
    g = np.clip(g, min_vals[1], max_vals[1])
    r = np.clip(r, min_vals[2], max_vals[2])
    # Merge the channels back together
    new_image = cv2.merge((b, g, r))
    return new_image

# Add Random Curved Lines to Image

In [ ]:
def add_random_curves(image, num_lines=10, thickness=2):
    h, w, _ = image.shape
    
    for _ in range(num_lines):
        # Generate 3 random points: start, control, and end
        # We use coordinates within the image bounds
        r0, c0 = random.randint(0, h-1), random.randint(0, w-1)
        r1, c1 = random.randint(0, h-1), random.randint(0, w-1) # Control point
        r2, c2 = random.randint(0, h-1), random.randint(0, w-1)
        
        # Calculate the curve coordinates
        # weight=1 creates a standard quadratic curve
        rr, cc = bezier_curve(r0, c0, r1, c1, r2, c2, weight=1)
        
        # Filter coordinates that might fall outside the image bounds
        valid = (rr >= 0) & (rr < h) & (cc >= 0) & (cc < w)
        rr, cc = rr[valid], cc[valid]
        
        # Pick a random color for the line
        color = [random.randint(0, 255) for _ in range(3)]
        
        # Draw the curve onto the image
        # Note: To handle thickness > 1, you'd usually dilate these points
        image[rr, cc] = color
        
    return image

# Shift Image

In [ ]:
def Shift(img,x,y,xl,yl,C=5):
    rows, cols, ch = img.shape
    pts1 = np.float32([[random.randint(-C,C),random.randint(-C,C)],
                       [cols-random.randint(-C,C),random.randint(-C,C)],
                       [random.randint(-C,C),rows-random.randint(-C,C)]])
    shear_amount = random.randint(-C,C)
    pts2 = np.float32([[random.randint(-C,C),random.randint(-C,C)],
                       [cols-random.randint(-C,C)+shear_amount,random.randint(-C,C)],
                       [random.randint(-C,C) - shear_amount, rows - random.randint(-C,C)]])
    M = cv2.getAffineTransform(pts1, pts2)
    warped_img = cv2.warpAffine(img, M, (cols, rows))
    res = np.copy(img)
    res[x:x+xl,y:y+yl,:] = warped_img[x:x+xl,y:y+yl,:]
    return res

# Deform Image

In [ ]:
def img_deformation(image, max_movement=10, smoothness=50):
    # Get the height and width of the image
    h, w, _ = image.shape
    # Generate random noise-based flow fields for X and Y directions
    flow_X = np.random.uniform(-1, 1, (h, w)).astype(np.float32)
    flow_Y = np.random.uniform(-1, 1, (h, w)).astype(np.float32)
    # Smooth the noise to ensure consistency across regions
    flow_X = gaussian_filter(flow_X, sigma=smoothness)
    flow_Y = gaussian_filter(flow_Y, sigma=smoothness)
    # Normalize the flow fields to the range of [-max_movement, max_movement]
    flow_X = cv2.normalize(flow_X, None, -max_movement, max_movement, cv2.NORM_MINMAX)
    flow_Y = cv2.normalize(flow_Y, None, -max_movement, max_movement, cv2.NORM_MINMAX)
    # Generate a meshgrid of coordinates (X, Y)
    X, Y = np.meshgrid(np.arange(w), np.arange(h))
    # Create a map that distorts the image based on the smoothed optical flow
    map_X = (X + flow_X).astype(np.float32)
    map_Y = (Y + flow_Y).astype(np.float32)
    # Warp the image using remap function
    warped_image = cv2.remap(image, map_X, map_Y, interpolation=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
    return warped_image

# Warp Each Pixel to Random Nearby Location

In [ ]:
def random_point_warp(image, max_movement=1):
    # Get the height and width of the image
    h, w, _ = image.shape
    # Generate a meshgrid of coordinates (X, Y)
    X, Y = np.meshgrid(np.arange(w), np.arange(h))
    # Generate random optical flow for X and Y directions
    flow_X = np.random.uniform(-max_movement, max_movement, (h, w)).astype(np.float32)
    flow_Y = np.random.uniform(-max_movement, max_movement, (h, w)).astype(np.float32)
    # Create a map that distorts the image based on the optical flow
    map_X = (X + flow_X).astype(np.float32)
    map_Y = (Y + flow_Y).astype(np.float32)
    # Warp the image using remap function
    warped_image = cv2.remap(image, map_X, map_Y, interpolation=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
    return warped_image

# Deform Image

In [ ]:
def img_deformation1(image, target_point=None, max_movement=10, falloff=100, smoothness=10):
    h, w, _ = image.shape
    # Default to center of image if no target specified
    if target_point is None:
        target_x, target_y = w // 2, h // 2
    else:
        target_x, target_y = target_point
    # Generate coordinate grid
    X, Y = np.meshgrid(np.arange(w), np.arange(h))
    # Compute displacement vectors toward the target
    disp_X = (target_x - X).astype(np.float32)
    disp_Y = (target_y - Y).astype(np.float32)
    # Compute distance to the target
    distance = np.sqrt(disp_X**2 + disp_Y**2)
    # Normalize vectors to unit direction
    unit_disp_X = disp_X / (distance + 1e-8)
    unit_disp_Y = disp_Y / (distance + 1e-8)
    # Apply falloff to control strength of displacement by distance
    movement_strength = np.exp(-distance / falloff) * max_movement
    # Final flow toward the target point
    flow_X = unit_disp_X * movement_strength
    flow_Y = unit_disp_Y * movement_strength
    # Optionally smooth the flow for natural warping
    flow_X = gaussian_filter(flow_X, sigma=smoothness)
    flow_Y = gaussian_filter(flow_Y, sigma=smoothness)
    # Create new sampling coordinates
    map_X = (X + flow_X).astype(np.float32)
    map_Y = (Y + flow_Y).astype(np.float32)
    # Warp the image
    warped_image = cv2.remap(image, map_X, map_Y, interpolation=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
    return warped_image

# Add Inconsistencies to List of Frames

In [ ]:
def AddInconsistencies_np(Frames, Steps, Op=None, prob=0.05):
    N = len(Frames)
    IncFrames = [f.copy() for f in Frames]
    Bin = [np.zeros((Frames[0].shape[0],Frames[0].shape[1])) for _ in range(N)]
    Size = (IncFrames[0].shape[0],IncFrames[0].shape[1])  # (height, width)
    if Steps > 0:
        steps = random.randint(1, Steps)
        for s in range(steps):
            x = random.randint(0, 5 * Size[0] // 6)
            y = random.randint(0, 5 * Size[1] // 6)
            xl = random.randint(10, Size[0] // 8)
            yl = random.randint(10, Size[1] // 8)
            for p in range(N):
                if random.random() <= prob and p != N // 2:
                    IncFrames[p], Bin[p] = AddOneInc(Bin[p], IncFrames[p], x=x, y=y, xl=xl, yl=yl, Op=Op,return_tensor=False, F_list=IncFrames,p_idx=p)
                elif p==N//2:
                    IncFrames[p], Bin[p] = AddOneInc(Bin[p], IncFrames[p], x=x, y=y, xl=xl, yl=yl, Op=Op,return_tensor=False, F_list=IncFrames,p_idx=p)
    return Bin,IncFrames

# Add Wavy Warping

In [ ]:
def Wavy(img,x1,y1,xl,yl):
    map_x = np.zeros((img.shape[0],img.shape[1]),np.float32)
    map_y = np.zeros((img.shape[0],img.shape[1]),np.float32)
    amplitude = random.uniform(1,20)
    frequency = random.uniform(1e-3,0.1)
    for i in range(img.shape[0]):
        for j in range(img.shape[1]):
            map_x[i, j] = j + amplitude * np.sin(frequency * i) 
            map_y[i, j] = i
    wavy_img = cv2.remap(img, cv2.merge([map_x, map_y]),None, cv2.INTER_LINEAR)
    res = np.copy(img)
    res[x1:x1+xl,y1:y1+yl,:] = wavy_img[x1:x1+xl,y1:y1+yl,:]
    return res

# Generate Vanishing Blob for Weighted Mask

In [ ]:
def generate_blob(h, w, center, target_area, seed=None):
    if seed is not None:
        np.random.seed(seed)

    cy, cx = center

    # --- 1. Fractal noise ---
    field = np.zeros((h, w), np.float32)
    for scale, weight in [(8, 1.0), (16, 0.8), (32, 0.6), (64, 0.4)]:
        noise = np.random.rand(h, w).astype(np.float32)
        noise = cv2.GaussianBlur(noise, (scale | 1, scale | 1), 0)
        field += weight * noise

    field /= field.max()

    # --- 2. Soft spatial bias ---
    y, x = np.ogrid[:h, :w]
    dist = np.sqrt((x - cx)**2 + (y - cy)**2)

    sigma = np.sqrt(target_area) * 0.6
    bias = np.exp(-(dist**2) / (2 * sigma**2))

    field = field * (0.6 + 0.8 * bias)

    # Normalize
    field = (field - field.min()) / (field.max() + 1e-8)

    # --- 3. Randomized threshold ---
    flat = np.sort(field.flatten())[::-1]

    jitter = np.random.uniform(0.5, 1.5)
    idx = int(min(len(flat)-1, target_area * jitter))
    threshold = flat[idx]

    binary = (field >= threshold).astype(np.uint8)

    # --- 4. Keep only connected component near center ---
    num_labels, labels = cv2.connectedComponents(binary)
    center_label = labels[cy, cx]
    binary = (labels == center_label).astype(np.uint8)

    # --- 5. Compute blob centroid (better than using initial center) ---
    ys, xs = np.where(binary > 0)
    if len(ys) > 0:
        cy_blob = ys.mean()
        cx_blob = xs.mean()
    else:
        cy_blob, cx_blob = cy, cx  # fallback

    # --- 6. Local smoothing (Gaussian blur) ---
    soft = cv2.GaussianBlur(binary.astype(np.float32), (5, 5), 0)

    # --- 7. Global Gaussian falloff (multiply) ---
    y, x = np.ogrid[:h, :w]
    sigma_falloff = np.sqrt(target_area) * 0.4

    gaussian = np.exp(-((x - cx_blob)**2 + (y - cy_blob)**2) / (2 * sigma_falloff**2))

    soft *= gaussian

    # Normalize final output
    soft = (soft - soft.min()) / (soft.max() + 1e-8)

    return binary, soft

# Generate Organic Weighted Mask

In [ ]:
def generate_organic_mask(h, w, y, x, yl, xl):
    # Create high-frequency noise for the 'blob' base
    small_h, small_w = max(4, yl // 8), max(4, xl // 8)
    noise = np.random.rand(small_h, small_w).astype(np.float32)
    
    # Upscale and smooth to create organic 'blobs'
    blob = cv2.resize(noise, (xl, yl), interpolation=cv2.INTER_CUBIC)
    blob = gaussian_filter(blob, sigma=min(xl, yl) // 12)
    
    # Normalize blob to 0-1 range
    blob = (blob - blob.min()) / (blob.max() - blob.min() + 1e-8)
    
    # Soft Alpha Mask (for blending)
    # We use a Sigmoid-like contrast boost to make the edges organic but distinct
    alpha_patch = np.clip((blob - 0.3) * 3, 0, 1)
    
    full_mask = np.zeros((h, w), dtype=np.float32)
    full_mask[y:y+yl, x:x+xl] = alpha_patch
    
    # Hard Binary Mask (for Ground Truth)
    # Any pixel with more than 5% modification is marked as 1
    hard_mask = (full_mask > 0.05).astype(np.float32)
    
    return full_mask, hard_mask

# Add One Inconsistency to a Single Frame

In [ ]:
def AddOneInc(Bin, F, Op=None, x=None, y=None, xl=None, yl=None, return_tensor=False, device=device, F_list=None,p_idx=None):
    F_np = to_numpy(F)
    Bin_np = Bin if isinstance(Bin, np.ndarray) else Bin.cpu().numpy()
    I_orig = F_np.astype(np.uint8).copy()
    I_modified = I_orig.copy()
    h, w = I_orig.shape[:2]

    xl = xl or random.randint(w // 20, w // 10)
    yl = yl or random.randint(h // 20, h // 10)
    x = x if x is not None else random.randint(0, w - xl)
    y = y if y is not None else random.randint(0, h - yl)
    if Op is None: Op = random.randint(0,23)

    alpha_mask, hard_label = generate_organic_mask(h, w, y, x, yl, xl)
    alpha_3ch = np.stack([alpha_mask] * 3, axis=-1)

    if Op==0: # Color Range
        # Extract the patch once to avoid repeated indexing
        patch = I_modified[y:y+yl, x:x+xl]
        
        # Calculate mins and maxs
        mins = patch.min(axis=(0,1))
        maxs = patch.max(axis=(0,1))
        
        # Ensure we are working with Python ints to avoid uint8 underflow (wrap-around)
        new_min = [max(0, min(255, random.randint(int(mins[i]) - 70, int(mins[i]) + 70))) for i in range(3)]
        new_max = [max(0, min(255, random.randint(int(maxs[i]) - 70, int(maxs[i]) + 70))) for i in range(3)]
        
        I_modified[y:y+yl, x:x+xl] = change_range_colors(patch, new_min, new_max)
        
    elif Op==1: #Add Solid Color
        I_modified[y:y+yl, x:x+xl] = [random.randint(0, 255) for _ in range(3)]
        
    elif Op==2: #Add Random Uniform Noise
        noise = np.random.randint(-255,255,I_modified[y:y+yl,x:x+xl].shape,dtype=np.int16)
        region = I_modified[y:y+yl,x:x+xl].astype(np.int16)
        I_modified[y:y+yl,x:x+xl] = np.clip(region+noise,0,255).astype(np.uint8)
        
    elif Op==3: #Add Blur
        ks = random.randrange(3,35,2)
        I_modified = cv2.GaussianBlur(I_modified,(ks,ks),0)

    elif Op==4: #Do a Random Kernel Convolution
        kernel = np.random.rand(5, 5) * 2.0 - 1.0
        I_modified = cv2.filter2D(I_modified, -1, kernel)
        
    elif Op==5: #Deformation
        I_modified[y:y+yl,x:x+xl]=img_deformation(I_modified[y:y+yl,x:x+xl],random.randint(10, 80),random.randint(10, 200))
        
    elif Op==6: #Rnd Point Warp
        I_modified[y:y+yl,x:x+xl]=random_point_warp(I_modified[y:y+yl,x:x+xl],random.randint(1, 20))        

    elif Op==7: #HSV Color Channels Changes
        hsv = cv2.cvtColor(I_modified, cv2.COLOR_BGR2HSV).astype(np.float32)
        channel = random.randint(0,2)
        hsv[:,:,channel] = (hsv[:,:,channel] + random.randint(40, 100)) % 180
        I_modified = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)
        
    elif Op==8: #Add Illumination Noise
        I_modified[y:y+yl,x:x+xl] = I_modified[y:y+yl,x:x+xl]*random.gauss(0.8,1/6)
        
    elif Op==9: #Deformation
        pt = (random.randint(0, xl), random.randint(0, xl))
        fall = random.randint(5, xl)
        sth = random.randint(5, 20)
        I_modified[y:y+yl,x:x+xl] = img_deformation1(I_modified[y:y+yl,x:x+xl], pt, xl, fall, sth)
        
    elif Op==10: #Add Shift Motion
        I_modified = Shift(I_modified,y,x,yl,xl)
    
    elif Op==11: #Add Wavy Motion
        I_modified = Wavy(I_modified,y,x,yl,xl)
        
    elif Op==12: #Add fine lines
        Lines = add_random_curves(np.zeros_like(I_modified),random.randint(20,100))
        I_modified += Lines
        
    elif Op == 13: #Affine Warping Blending
        shift_x, shift_y = (-1)**(random.randint(1,2))*random.randint(10, 50), (-1)**(random.randint(1,2))*random.randint(10, 50)
        echo_patch = I_orig[y:y+yl, x:x+xl].copy()
        M = np.float32([[3*random.random()-1.5, 2*random.random(), shift_x], [2*random.random(), 3*random.random()-1.5, shift_y]])
        echo_patch = cv2.warpAffine(echo_patch, M, (xl, yl), borderMode=cv2.BORDER_REFLECT)
        alfa = random.random()
        I_modified[y:y+yl, x:x+xl] = cv2.addWeighted(I_modified[y:y+yl, x:x+xl], alfa, echo_patch, 1-alfa, 0)
        
    elif Op == 14: # Local Patch Blending
        offset_y = random.randint(-50, 50)
        offset_x = random.randint(-50, 50)
        src_y = np.clip(y + offset_y, 0, h - yl)
        src_x = np.clip(x + offset_x, 0, w - xl)
        near_patch = I_orig[src_y:src_y+yl, src_x:src_x+xl]
        alfa = random.random()
        I_modified[y:y+yl, x:x+xl] = cv2.addWeighted(I_modified[y:y+yl, x:x+xl], alfa, near_patch, 1-alfa, 0)
        
    elif Op == 15: # Compression Blocks
        block_size = random.choice([2, 4, 8])
        down_w = max(1, xl // block_size)
        down_h = max(1, yl // block_size)
        small = cv2.resize(I_modified[y:y+yl, x:x+xl], (down_w, down_h), interpolation=cv2.INTER_LINEAR)
        I_modified[y:y+yl, x:x+xl] = cv2.resize(small, (xl, yl), interpolation=cv2.INTER_NEAREST)

    elif Op == 16:  # Cross-Frame Texture Pollution (Fixed)
        if F_list is not None and len(F_list) > 1 and p_idx is not None:
            # Pick a random source frame that is NOT the current processing frame
            chosen_idx = random.choice([i for i in range(len(F_list)) if i != p_idx])
            foreign_frame = to_numpy(F_list[chosen_idx]).astype(np.uint8)
            I_modified[y:y+yl, x:x+xl] = foreign_frame[y:y+yl, x:x+xl]

    elif Op == 17:  # Irregular Fluid Warp (Boiling Artifact)
        # Create unique micro-displacement fields representing structural flow-hallucinations
        xx, yy = np.meshgrid(np.arange(xl), np.arange(yl))
        noise_field_x = np.sin(xx / 4.0) * random.randint(4, 15)
        noise_field_y = np.cos(yy / 4.0) * random.randint(4, 15)
        map_x = (xx + noise_field_x).astype(np.float32)
        map_y = (yy + noise_field_y).astype(np.float32)
        I_modified[y:y+yl, x:x+xl] = cv2.remap(I_modified[y:y+yl, x:x+xl], map_x, map_y, cv2.INTER_CUBIC, borderMode=cv2.BORDER_REFLECT)
        
    elif Op == 18: # Frequency Distortion / Micro-Datamoshing (Idea 1)
        # Downsample aggressively to simulate reconstruction failure, then add DCT-like block noise
        patch = I_modified[y:y+yl, x:x+xl].astype(np.float32)
        small = cv2.resize(patch, (max(4, xl // 4), max(4, yl // 4)), interpolation=cv2.INTER_NEAREST)
        
        # Add frequency block artifacts
        block_noise = np.random.choice([-15, 0, 15], size=small.shape, p=[0.2, 0.6, 0.2])
        small = np.clip(small + block_noise, 0, 255).astype(np.uint8)
        
        # Scale back up with blocky artifacts
        I_modified[y:y+yl, x:x+xl] = cv2.resize(small, (xl, yl), interpolation=cv2.INTER_NEAREST)

    elif Op == 19: # Morphing / Dissolving Hallucination (Idea 2)
        # Distort the patch heavily while blending it with a foreign organic pattern
        patch = I_modified[y:y+yl, x:x+xl]
        
        # Create a dynamic fluid warp field
        xx, yy = np.meshgrid(np.arange(xl), np.arange(yl))
        map_x = (xx + np.sin(yy / 3.0) * 15).astype(np.float32)
        map_y = (yy + np.cos(xx / 3.0) * 15).astype(np.float32)
        warped_patch = cv2.remap(patch, map_x, map_y, cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
        
        # Generate a procedural AI "hallucination" texture (fractal noise)
        foreign_texture = np.zeros_like(patch)
        for i in range(3):
            noise = np.random.randint(0, 255, (yl, xl), dtype=np.uint8)
            foreign_texture[:, :, i] = cv2.GaussianBlur(noise, (15, 15), 0)
            
        # Blend the morphing patch with the hallucinated texture
        I_modified[y:y+yl, x:x+xl] = cv2.addWeighted(warped_patch, 0.6, foreign_texture, 0.4, 0)

    elif Op == 20: # Global Semantic Color Flashing (Idea 3)
        # Grab a target color mask from the entire frame (e.g., matching a background or clothing tone)
        hsv = cv2.cvtColor(I_modified, cv2.COLOR_BGR2HSV)
        
        # Randomly choose a target channel color range to isolate semantically
        rand_hue = random.randint(0, 180)
        lower_bound = np.array([max(0, rand_hue - 20), 40, 40])
        upper_bound = np.array([min(180, rand_hue + 20), 255, 255])
        
        semantic_mask = cv2.inRange(hsv, lower_bound, upper_bound)
        
        # Apply a dramatic Hue shift exclusively to that semantic mask
        hsv_modified = hsv.copy().astype(np.int32)
        hsv_modified[:, :, 0] = (hsv_modified[:, :, 0] + random.randint(30, 120)) % 180
        hsv_modified = np.clip(hsv_modified, 0, 255).astype(np.uint8)
        
        img_semantic = cv2.cvtColor(hsv_modified, cv2.COLOR_HSV2BGR)
        
        # Overwrite only the semantically grouped pixels across the entire image
        I_modified = np.where(np.stack([semantic_mask] * 3, axis=-1) > 0, img_semantic, I_modified)
        
        # Force the tracking hard_label to capture this frame-wide change
        hard_label = (semantic_mask > 0).astype(np.float32)

    elif Op == 21: # High-Contrast Edge-Targeted Artifact (Idea 5)
        # Find high-gradient details (faces, boundaries, text) where AI models struggle
        gray = cv2.cvtColor(I_orig, cv2.COLOR_BGR2GRAY)
        edges = cv2.Canny(gray, 50, 150)
        edge_y, edge_x = np.where(edges > 0)
        
        # If the image has complex details, snap our patch directly onto one of those zones
        if len(edge_y) > 0:
            idx = random.randint(0, len(edge_y) - 1)
            y = np.clip(edge_y[idx] - yl // 2, 0, h - yl)
            x = np.clip(edge_x[idx] - xl // 2, 0, w - xl)
            
            # Re-generate the organic masks relative to the updated high-contrast coordinates
            alpha_mask, hard_label = generate_organic_mask(h, w, y, x, yl, xl)
            alpha_3ch = np.stack([alpha_mask] * 3, axis=-1)
            
        # Execute an advanced, blurred texture smash on this detailed region
        patch = I_modified[y:y+yl, x:x+xl]
        kernel = np.ones((5, 5), np.float32) / -25.0
        kernel[2, 2] = 2.0  # Harsh, unnatural AI over-sharpening
        sharpened = cv2.filter2D(patch, -1, kernel)
        I_modified[y:y+yl, x:x+xl] = cv2.GaussianBlur(sharpened, (5, 5), 0)

    elif Op == 22: # Temporal Motion Ghosting / Frame Lag
        if F_list is not None and p_idx is not None and p_idx > 0:
            # Grab the patch from the previous chronological frame to create a trailing ghost artifact
            prev_frame = to_numpy(F_list[p_idx - 1]).astype(np.uint8)
            ghost_patch = prev_frame[y:y+yl, x:x+xl]
            
            # Blend the previous frame's memory back into the current frame
            I_modified[y:y+yl, x:x+xl] = cv2.addWeighted(I_modified[y:y+yl, x:x+xl], 0.5, ghost_patch, 0.5, 0)

    elif Op == 23: # Temporal Flicker / Coherence Breakdown
        if F_list is not None and p_idx is not None:
            mid_idx = len(F_list) // 2
            
            if p_idx == mid_idx:
                # The target middle frame gets a highly distorted warp representation
                xx, yy = np.meshgrid(np.arange(xl), np.arange(yl))
                map_x = (xx + np.sin(yy / 2.0) * 10).astype(np.float32)
                map_y = (yy + np.cos(xx / 2.0) * 10).astype(np.float32)
                
                # FIX: Added borderMode=cv2.BORDER_REFLECT to stop black edge pixels from bleeding in
                I_modified[y:y+yl, x:x+xl] = cv2.remap(
                    I_modified[y:y+yl, x:x+xl], 
                    map_x, map_y, 
                    cv2.INTER_LINEAR, 
                    borderMode=cv2.BORDER_REFLECT
                )
            else:
                # Cast to float32 first to prevent uint8 math overflows
                patch_float = I_modified[y:y+yl, x:x+xl].astype(np.float32)
                flash_patch = np.clip(patch_float * 1.8, 0, 255).astype(np.uint8)
                I_modified[y:y+yl, x:x+xl] = flash_patch

    # 3. Alpha Blending for Realism
    I_final = (I_orig * (1 - alpha_3ch) + I_modified * alpha_3ch).astype(np.uint8)

    # 4. Update Binary Ground Truth (Hard Mask)
    # We use bitwise OR to keep existing inconsistencies in 'Bin' if multiple are added    
    if Op==12:
        b_Lines = (Lines.sum(axis=-1)>0)*1.0
        final_bin = np.logical_and(hard_label,b_Lines).astype(np.float32)
        final_bin = np.logical_or(Bin_np,final_bin).astype(np.float32)
    else:
        final_bin = np.logical_or(Bin_np, hard_label).astype(np.float32)

    
    if return_tensor:
        return to_tensor(I_final, device), torch.tensor(final_bin, dtype=torch.float32).to(device)
    return I_final, final_bin